In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def mgd_tx60l(q):
    # Conversion en radians et calcul des cos/sin
    q = np.array(q, dtype=float)
    c = np.cos(q)
    s = np.sin(q)
    
    # T01 : translation  Z (0.375) + Rot Z (q1)
    T01 = np.array([[c[0], -s[0], 0, 0],
                    [s[0],  c[0], 0, 0],
                    [   0,     0, 1, 0.375],
                    [   0,     0, 0, 1]])

    # T12 : Rot X (90°) + Rot Z (q2)
    T12 = np.array([[c[1], -s[1], 0, 0],
                    [   0,     0,-1, 0],
                    [s[1],  c[1], 0, 0],
                    [   0,     0, 0, 1]])

    # T23 : translation X (0.02) + translation Y (0.400) + Rot Z (q3)
    T23 = np.array([[c[2], -s[2], 0, 0.020],
                    [s[2],  c[2], 0, 0.400],
                    [   0,     0, 1, 0],
                    [   0,     0, 0, 1]])

    # T34 : translation Y (0.450) + Rot X (-90°) + Rot Z (q4)
    T34 = np.array([[c[3], -s[3], 0, 0],
                    [   0,     0, 1, 0.450],
                    [-s[3],-c[3], 0, 0],
                    [   0,     0, 0, 1]])

    # T45 : Rot X (90°) + Rot Z (q5)
    T45 = np.array([[c[4], -s[4], 0, 0],
                    [   0,     0,-1, 0],
                    [s[4],  c[4], 0, 0],
                    [   0,     0, 0, 1]])

    # T56 : translation Y (0.070) + Rot X (-90°) + Rot Z (q6)
    T56 = np.array([[c[5], -s[5], 0, 0],
                    [   0,     0, 1, 0.070],
                    [-s[5],-c[5], 0, 0],
                    [   0,     0, 0, 1]])

    return T01 @ T12 @ T23 @ T34 @ T45 @ T56

T_resultat = mgd_tx60l([0, 0, 0, 0, 0, 0])
print(np.round(T_resultat, 4))

[[1.    0.    0.    0.02 ]
 [0.    1.    0.    0.   ]
 [0.    0.    1.    1.295]
 [0.    0.    0.    1.   ]]


In [3]:
q = [np.pi/2, 0, 0, 0, 0, 0]
T_resultat = mgd_tx60l(q)
print("q =",q)
print(np.round(T_resultat, 4))

q = [0, np.pi/2, 0, 0, 0, 0]
T_resultat = mgd_tx60l(q)
print("q =",q)
print(np.round(T_resultat, 4))

q = [1.5707963267948966, 0, 0, 0, 0, 0]
[[ 0.    -1.     0.     0.   ]
 [ 1.     0.     0.     0.02 ]
 [ 0.     0.     1.     1.295]
 [ 0.     0.     0.     1.   ]]
q = [0, 1.5707963267948966, 0, 0, 0, 0]
[[ 0.     0.    -1.    -0.92 ]
 [ 0.     1.     0.     0.   ]
 [ 1.     0.     0.     0.395]
 [ 0.     0.     0.     1.   ]]


In [4]:
def mgi_numerique(
    mgd_func, pos_cible, q_init, max_iter=100, tol=1e-5, damping=0.01
):
    """Calcul du MGI par algorithme itératif.

    :param mgd_func: Fonction mgd(q) renvoyant (pos, quat)
    :param pos_cible: Position visée [x, y, z]
    :param q_init: Configuration articulaire initiale (liste ou array)
    :return: (q_sol, succes)
    """
    q = np.array(q_init, dtype=float)
    target = np.array(pos_cible, dtype=float)

    for i in range(max_iter):
        pos_courante, _ = mgd_func(q)
        err = target - np.array(pos_courante)

        # Critère d'arrêt
        if np.linalg.norm(err) < tol:
            return q, True

        # Calcul numérique de la Jacobienne de position
        n = len(q)
        J = np.zeros((3, n))
        eps = 1e-6
        for j in range(n):
            q_eps = np.copy(q)
            q_eps[j] += eps
            pos_eps, _ = mgd_func(q_eps)
            J[:, j] = (np.array(pos_eps) - np.array(pos_courante)) / eps

        # Résolution par pseudo-inverse amortie
        J_t = J.T
        dq = J_t @ np.linalg.inv(J @ J_t + (damping**2) * np.eye(3)) @ err
        q += dq

    return q, False

# Wrapper pour extraire la position de la matrice 4x4
def mgd_pos_tx60l(q):
    T = mgd_tx60l(q)
    pos = T[:3, 3]  # Extraire [x, y, z] (colonne 3, lignes 0 à 2)
    return pos, T  # Renvoie un tuple (position, matrice_T)


# --- EXECUTION MGI ---

# 1. Cible et point de départ
target_pos = [0.6, 0.2, 0.3]
q_start = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

# 2. Calcul du MGI
q_sol, succes = mgi_numerique(mgd_pos_tx60l, target_pos, q_start)

# 3. Validation
if succes:
    pos_recalculee, _ = mgd_pos_tx60l(q_sol)
    erreur = np.linalg.norm(np.array(target_pos) - np.array(pos_recalculee))
    print(f"MGI Convergé : q = {np.round(q_sol, 4)}")
    print(f"Erreur de bouclage : {erreur:.6e} m")
else:
    print("Échec de convergence du MGI.")

MGI Convergé : q = [-15.3224 -43.1656  45.5498   7.6811   5.6589   0.    ]
Erreur de bouclage : 1.305340e-06 m
